[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/08-ai-engineering/05_serving.ipynb)

*AI Engineering · Notebook 5 of 10*

# Serving LLMs: Inference Optimisation

What it takes to run models yourself: memory for weights and the KV cache, batching (static vs continuous), prefill vs decode, speculative decoding, quantisation, paged KV memory and prefix sharing, scheduling, rate limits, fallbacks and the self-host vs API cost decision.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Serving an LLM is running a busy kitchen: memory is counter space, batching cooks many orders at once, and the KV cache is the prep you don't redo.

**What interviewers really test here:** Weight and KV-cache memory, continuous batching, prefill vs decode, quantisation, speculative decoding and self-host vs API economics.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. API or your own GPUs? *(Basics)*
- [ ] 2. Memory: weights and the KV cache *(Core)*
- [ ] 3. Batching, prefill and decode *(Core)*
- [ ] 4. Making each token cheaper *(Core)*
- [ ] 5. Running it in production *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [ ]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
import numpy as np
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Key serving metrics

| Metric | Meaning | Driven by |
| --- | --- | --- |
| TTFT | time to first token | queueing + prefill (prompt length) |
| TPOT / ITL | time per output token | decode speed, batch size |
| Throughput | total tokens/s across users | batching, GPU memory, quantisation |
| Goodput | requests/s that meet the latency target | everything above |
| Cost per 1M tokens | GPU cost ÷ tokens produced | throughput and utilisation |

### Why decode is memory-bound

Generating one token for one sequence needs every weight read from GPU memory once, but does relatively little arithmetic with it. Memory bandwidth, not compute, is the limit. Batching reuses each weight read for many sequences, which is why it raises throughput so much, and why smaller (quantised) weights speed decode up.

---
# Part 2: Lessons, step by step

## Lesson 1: API or your own GPUs?

*Level: Basics*

Most teams start with a hosted API: no infrastructure, the best models, pay per token. Teams **self-host open-weight models** (Llama, Qwen, Mistral, Gemma and others) when they need data to stay inside their network, very high volume at low cost per token, a fine-tuned small model, or offline use. Self-hosting means you now own memory, batching, latency and uptime — the subject of this topic.



| Tool | Used for |
| --- | --- |
| **vLLM**, **SGLang**, TensorRT-LLM | high-throughput GPU servers with continuous batching |
| **llama.cpp**, **Ollama** | laptops and CPUs, quantised models |
| Hugging Face TGI / Inference Endpoints, cloud model gardens | managed hosting of open models |



```python
pip install vllm
vllm serve Qwen/Qwen2.5-7B-Instruct --max-model-len 8192
curl localhost:8000/v1/chat/completions -H "Content-Type: application/json" \
  -d '{"model": "Qwen/Qwen2.5-7B-Instruct", "messages": [{"role": "user", "content": "Hi"}]}'
```

**✏️ Your turn:** you now have what you need for [Q16](#q16), [Q5](#q5). Try one now, or carry on and do them in Part 3.

## Lesson 2: Memory: weights and the KV cache

*Level: Core*

GPU memory holds two big things:




- **Weights**: parameters × bytes each. An 8B model is 16 GB in bf16, 8 GB in int8, 4 GB in int4.
- **KV cache**: while generating, each layer keeps the key and value vectors of every previous token so it doesn't recompute them. Size = 2 × layers × KV heads × head size × bytes × tokens × sequences. It grows with context length *and* with the number of users, and is usually what limits how many requests a GPU can serve at once.



Grouped-query attention (fewer KV heads than query heads) shrinks the KV cache a lot — compare 8 KV heads with 32 in the questions.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q8](#q8), [Q9](#q9), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 3: Batching, prefill and decode

*Level: Core*

Generation has two phases. **Prefill** reads the whole prompt in parallel — fast per token, and it sets the time to first token. **Decode** produces one token at a time; each step reads all the weights from memory, so a single sequence leaves the GPU mostly idle.



The fix is **batching**: decode many sequences per step. **Static batching** waits for the slowest sequence in a batch before starting the next. **Continuous batching** (vLLM, SGLang) slots a new request in the moment any sequence finishes, keeping the GPU busy — often several times more throughput.



Bigger batches raise total tokens per second but lower each user's speed; pick the batch size that meets your per-user latency target. Scheduling also matters: short jobs first lowers average waiting time.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [ ]:
import matplotlib.pyplot as plt


def batching_chart():
    lengths = [10, 200, 15, 30, 190, 20, 25, 12]
    import heapq
    static, cont = [], []
    for slots in (1, 2, 4, 8):
        static.append(sum(max(lengths[i:i + slots]) for i in range(0, len(lengths), slots)))
        heap = [0] * slots
        for L in lengths:
            heapq.heappush(heap, heapq.heappop(heap) + L)
        cont.append(max(heap))
    fig, ax = plt.subplots(figsize=(5, 2.6))
    ax.plot([1, 2, 4, 8], static, marker="o", label="static batching")
    ax.plot([1, 2, 4, 8], cont, marker="o", label="continuous batching")
    ax.set_xlabel("slots per step"); ax.set_ylabel("steps to finish all"); ax.legend()


batching_chart()

**✏️ Your turn:** you now have what you need for [Q20](#q20), [Q2](#q2), [Q4](#q4), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 4: Making each token cheaper

*Level: Core*

- **Quantisation**: store weights in 8 or 4 bits instead of 16. Less memory and faster decode (it's memory-bound), with a small quality loss. Per-group scales handle outlier weights much better than one scale for the whole tensor (methods like GPTQ, AWQ). KV caches can be quantised too.
- **Speculative decoding**: a small draft model guesses several tokens; the big model checks them all in one step and keeps the matching prefix. Same output, often 2–3× faster when the draft is usually right.
- **Paged KV memory** (PagedAttention): allocate the KV cache in small blocks as sequences grow instead of reserving the maximum length, so far more sequences fit.
- **Prefix caching**: share the KV blocks of a common system prompt across requests.

**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q13](#q13), [Q10](#q10), [Q11](#q11), [Q14](#q14), [Q3](#q3). Try one now, or carry on and do them in Part 3.

## Lesson 5: Running it in production

*Level: Advanced*

- **Capacity**: by Little's law, requests in flight = arrival rate × time per request; size the fleet with headroom.
- **Rate limits**: a token bucket per customer stops one user starving the rest.
- **Fallbacks**: if the main model errors or times out, retry on a secondary model or provider.
- **Multi-tenant fine-tunes**: serve one base model with many small LoRA adapters instead of one full copy per customer.
- **Measure**: time to first token, tokens per second per user, total throughput, GPU utilisation and cost per million tokens.

**✏️ Your turn:** you now have what you need for [Q15](#q15), [Q18](#q18), [Q6](#q6), [Q7](#q7). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Memory for the Weights

🟢 Warm-up · *Will it fit on the GPU?*

Return the GB needed to hold `params` parameters (in billions) at each dtype in `dtypes`, rounded to 1 (1 GB = 10⁹ bytes).

**Examples**

- `params = 8, dtypes = ['fp32', 'bf16', 'int8', 'int4']` → `{'fp32': 32, 'bf16': 16, 'int8': 8, 'int4': 4.0}`
- `params = 70, dtypes = ['bf16', 'int4']` → `{'bf16': 140, 'int4': 35.0}`

In [ ]:
BYTES = {"fp32": 4, "bf16": 2, "fp8": 1, "int8": 1, "int4": 0.5}

def weight_gb(params, dtypes):
    """
    Return the GB needed to hold params parameters (in billions) at each dtype
    in dtypes, rounded to 1 (1 GB = 10⁹ bytes).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(weight_gb, [
    ((8, ['fp32', 'bf16', 'int8', 'int4']), {'fp32': 32, 'bf16': 16, 'int8': 8, 'int4': 4.0}),
    ((70, ['bf16', 'int4']), {'bf16': 140, 'int4': 35.0}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with a bytes table.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dict comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop with a bytes table.

```python
def weight_gb(params, dtypes):
    out = {}
    for d in dtypes:
        out[d] = round(params * 1e9 * BYTES[d] / 1e9, 1)
    return out
```

**Cost:** O(d) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dict comprehension.

```python
def weight_gb(params, dtypes):
    return {d: round(params * BYTES[d], 1) for d in dtypes}
```

**Cost:** O(d) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with a bytes table. That costs O(d) time and O(d) space."

"I can do better: A dict comprehension. That's O(d) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/memory-for-the-weights)

<a id="q2"></a>
### Q2. Prefill vs Decode

🟢 Warm-up · *Where the time goes*

Prefill processes the whole prompt in parallel at `prefill_tps` tokens/s; decode makes one token at a time at `decode_tps` tokens/s. Return `(time to first token, total time)` in seconds, rounded to 2.

**Examples**

- `prompt = 4000, output = 300, prefill_tps = 8000, decode_tps = 60` → `(0.5, 5.5)`
- `prompt = 200, output = 1500, prefill_tps = 8000, decode_tps = 60` → `(0.03, 25.02)`

In [ ]:
def prefill_decode(prompt, output, prefill_tps, decode_tps):
    """
    Prefill processes the whole prompt in parallel at prefill_tps tokens/s;
    decode makes one token at a time at decode_tps tokens/s. Return (time to
    first token, total time) in seconds, rounded to 2.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(prefill_decode, [
    ((4000, 300, 8000, 60), (0.5, 5.5)),
    ((200, 1500, 8000, 60), (0.03, 25.02)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two divisions.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The same, as one tuple expression.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Two divisions.

```python
def prefill_decode(prompt, output, prefill_tps, decode_tps):
    ttft = prompt / prefill_tps
    total = ttft + output / decode_tps
    return (round(ttft, 2), round(total, 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** The same, as one tuple expression.

```python
def prefill_decode(prompt, output, prefill_tps, decode_tps):
    ttft = prompt / prefill_tps
    return (round(ttft, 2), round(ttft + output / decode_tps, 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two divisions. That costs O(1) time and O(1) space."

"I can do better: The same, as one tuple expression. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/prefill-vs-decode)

<a id="q3"></a>
### Q3. Share the System Prompt's Cache

🟢 Warm-up · *Prefix caching*

Every request starts with the same `prefix`-token system prompt. With prefix sharing its KV blocks are stored once. Return `(blocks without sharing, blocks with sharing)` for requests whose own extra tokens are `extras` (blocks of `block` tokens; the prefix fills whole blocks).

**Examples**

- `prefix = 2048, extras = [100, 400, 30, 900], block = 16` → `(603, 219)`

In [ ]:
def shared_prefix(prefix, extras, block):
    """
    Every request starts with the same prefix-token system prompt. With prefix
    sharing its KV blocks are stored once. Return (blocks without sharing,
    blocks with sharing) for requests whose own extra tokens are extras (blocks
    of block tokens; the prefix fills whole blocks).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(shared_prefix, [
    ((2048, [100, 400, 30, 900], 16), (603, 219)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count blocks per request.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sums with ceiling division.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count blocks per request.

```python
def shared_prefix(prefix, extras, block):
    p_blocks = prefix // block
    without = 0
    with_sharing = p_blocks
    for e in extras:
        e_blocks = e // block + (1 if e % block else 0)
        without += p_blocks + e_blocks
        with_sharing += e_blocks
    return (without, with_sharing)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Sums with ceiling division.

```python
def shared_prefix(prefix, extras, block):
    p = prefix // block
    own = sum(-(-e // block) for e in extras)
    return (p * len(extras) + own, p + own)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count blocks per request. That costs O(n) time and O(1) space."

"I can do better: Sums with ceiling division. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/share-the-system-prompts-cache)

<a id="q4"></a>
### Q4. Throughput vs Per-User Speed

🟢 Warm-up · *Picking a batch size*

`BATCH_TABLE` rows are `(batch size, tokens/s per user, total tokens/s)`. Pick the batch size with the highest total throughput while each user still gets at least `min_user_tps`.

**Examples**

- `min_user_tps = 40` → `32`
- `min_user_tps = 10` → `128`
- `min_user_tps = 100` → `None`

In [ ]:
BATCH_TABLE = [(1, 95, 95), (8, 80, 640), (32, 52, 1664), (64, 31, 1984), (128, 17, 2176)]

def pick_batch(min_user_tps):
    """
    BATCH_TABLE rows are (batch size, tokens/s per user, total tokens/s). Pick
    the batch size with the highest total throughput while each user still gets
    at least min_user_tps.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pick_batch, [
    ((40,), 32),
    ((10,), 128),
    ((100,), None),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and keep the best valid row.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max` over a filtered generator, with a default.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop and keep the best valid row.

```python
def pick_batch(min_user_tps):
    best = None
    for b, user, total in BATCH_TABLE:
        if user >= min_user_tps and (best is None or total > best[2]):
            best = (b, user, total)
    return best[0] if best else None
```

**Cost:** O(r) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `max` over a filtered generator, with a default.

```python
def pick_batch(min_user_tps):
    ok = [r for r in BATCH_TABLE if r[1] >= min_user_tps]
    return max(ok, key=lambda r: r[2])[0] if ok else None
```

**Cost:** O(r) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and keep the best valid row. That costs O(r) time and O(1) space."

"I can do better: `max` over a filtered generator, with a default. That's O(r) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/throughput-vs-per-user-speed)

<a id="q5"></a>
### Q5. Cost per Million Tokens, Self-Hosted

🟢 Warm-up · *Unit economics*

A GPU costs `usd_per_hour` and produces `tps` output tokens per second at `util` average utilisation. Return dollars per million output tokens, rounded to 3.

**Examples**

- `usd_per_hour = 2.5, tps = 2000, util = 0.6` → `0.579`
- `usd_per_hour = 2.5, tps = 2000, util = 0.1` → `3.472`

In [ ]:
def self_host_cost(usd_per_hour, tps, util):
    """
    A GPU costs usd_per_hour and produces tps output tokens per second at util
    average utilisation. Return dollars per million output tokens, rounded to 3.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(self_host_cost, [
    ((2.5, 2000, 0.6), 0.579),
    ((2.5, 2000, 0.1), 3.472),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Tokens per hour, then divide.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One expression.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Tokens per hour, then divide.

```python
def self_host_cost(usd_per_hour, tps, util):
    tokens_per_hour = tps * 3600 * util
    return round(usd_per_hour / tokens_per_hour * 1_000_000, 3)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One expression.

```python
def self_host_cost(usd_per_hour, tps, util):
    return round(usd_per_hour * 1e6 / (tps * 3600 * util), 3)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Tokens per hour, then divide. That costs O(1) time and O(1) space."

"I can do better: One expression. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/cost-per-million-tokens-self-hosted)

<a id="q6"></a>
### Q6. Fall Back When a Model Fails

🟢 Warm-up · *Resilience*

Try models in order; `status[model]` says whether it succeeds and `latency[model]` how long the attempt takes (a failure still costs its latency). Return `(model that answered or None, total seconds)`.

**Examples**

- `models = ['primary', 'secondary', 'small'], status = {'primary': False, 'secondary': True, 'small': True}, latency = {'primary': 4.0, 'secondary': 2.5, 'small': 1.0}` → `('secondary', 6.5)`
- `models = ['a', 'b'], status = {'a': False, 'b': False}, latency = {'a': 1.0, 'b': 1.5}` → `(None, 2.5)`

In [ ]:
def fallback(models, status, latency):
    """
    Try models in order; status[model] says whether it succeeds and
    latency[model] how long the attempt takes (a failure still costs its
    latency). Return (model that answered or None, total seconds).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fallback, [
    ((['primary', 'secondary', 'small'], {'primary': False, 'secondary': True, 'small': True}, {'primary': 4.0, 'secondary': 2.5, 'small': 1.0}), ('secondary', 6.5)),
    ((['a', 'b'], {'a': False, 'b': False}, {'a': 1.0, 'b': 1.5}), (None, 2.5)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop that adds latency and stops on success.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same loop with `for … else`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop that adds latency and stops on success.

```python
def fallback(models, status, latency):
    total = 0.0
    for m in models:
        total += latency[m]
        if status[m]:
            return (m, total)
    return (None, total)
```

**Cost:** O(m) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same loop with `for … else`.

```python
def fallback(models, status, latency):
    total = 0.0
    for m in models:
        total += latency[m]
        if status[m]:
            break
    else:
        m = None
    return (m, total)
```

**Cost:** O(m) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop that adds latency and stops on success. That costs O(m) time and O(1) space."

"I can do better: Same loop with `for … else`. That's O(m) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/fall-back-when-a-model-fails)

<a id="q7"></a>
### Q7. Many Fine-Tunes on One GPU

🟢 Warm-up · *Serving LoRA adapters*

Serving `n` customers' fine-tuned models: full copies cost `base_gb` each, while LoRA serving keeps one base model plus an adapter of `adapter_mb` MB per customer. Return `(GB for full copies, GB with adapters)`, rounded to 2.

**Examples**

- `n = 50, base_gb = 16, adapter_mb = 40` → `(800, 18.0)`

In [ ]:
def adapters(n, base_gb, adapter_mb):
    """
    Serving n customers' fine-tuned models: full copies cost base_gb each, while
    LoRA serving keeps one base model plus an adapter of adapter_mb MB per
    customer. Return (GB for full copies, GB with adapters), rounded to 2.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(adapters, [
    ((50, 16, 40), (800, 18.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two products.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One tuple expression.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Two products.

```python
def adapters(n, base_gb, adapter_mb):
    full = n * base_gb
    lora = base_gb + n * adapter_mb / 1000
    return (round(full, 2), round(lora, 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One tuple expression.

```python
def adapters(n, base_gb, adapter_mb):
    return (round(n * base_gb, 2), round(base_gb + n * adapter_mb / 1000, 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two products. That costs O(1) time and O(1) space."

"I can do better: One tuple expression. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/many-fine-tunes-on-one-gpu)

<a id="q8"></a>
### Q8. How Big Is the KV Cache?

🟡 Interview standard · *Memory grows with tokens*

The KV cache stores a key and a value vector per layer, per KV head, per token. Return GB (rounded to 2) for `batch` sequences of `tokens` tokens.

**Examples**

- `layers = 32, kv_heads = 8, head_dim = 128, dtype_bytes = 2, tokens = 8192, batch = 1` → `1.07`
- `layers = 32, kv_heads = 8, head_dim = 128, dtype_bytes = 2, tokens = 8192, batch = 16` → `17.18`
- `layers = 32, kv_heads = 32, head_dim = 128, dtype_bytes = 2, tokens = 8192, batch = 1` → `4.29`

In [ ]:
def kv_gb(layers, kv_heads, head_dim, dtype_bytes, tokens, batch):
    """
    The KV cache stores a key and a value vector per layer, per KV head, per
    token. Return GB (rounded to 2) for batch sequences of tokens tokens.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(kv_gb, [
    ((32, 8, 128, 2, 8192, 1), 1.07),
    ((32, 8, 128, 2, 8192, 16), 17.18),
    ((32, 32, 128, 2, 8192, 1), 4.29),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Multiply step by step: per token per layer, then up.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One product.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Multiply step by step: per token per layer, then up.

```python
def kv_gb(layers, kv_heads, head_dim, dtype_bytes, tokens, batch):
    per_token_layer = 2 * kv_heads * head_dim * dtype_bytes
    per_token = per_token_layer * layers
    return round(per_token * tokens * batch / 1e9, 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One product.

```python
def kv_gb(layers, kv_heads, head_dim, dtype_bytes, tokens, batch):
    return round(2 * layers * kv_heads * head_dim * dtype_bytes * tokens * batch / 1e9, 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Multiply step by step: per token per layer, then up. That costs O(1) time and O(1) space."

"I can do better: One product. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/how-big-is-the-kv-cache)

<a id="q9"></a>
### Q9. How Many Users at Once?

🟡 Interview standard · *Concurrency on one GPU*

A GPU has `gpu_gb` memory; weights take `weights_gb` and the runtime reserves 10% of the GPU. Each sequence needs `kv_mb_per_token` MB per token, for `tokens` tokens. Return how many sequences fit (rounded down).

**Examples**

- `gpu_gb = 80, weights_gb = 16, kv_mb_per_token = 0.125, tokens = 8192` → `54`
- `gpu_gb = 80, weights_gb = 16, kv_mb_per_token = 0.125, tokens = 2048` → `218`
- `gpu_gb = 24, weights_gb = 16, kv_mb_per_token = 0.125, tokens = 8192` → `5`

In [ ]:
def max_sequences(gpu_gb, weights_gb, kv_mb_per_token, tokens):
    """
    A GPU has gpu_gb memory; weights take weights_gb and the runtime reserves
    10% of the GPU. Each sequence needs kv_mb_per_token MB per token, for tokens
    tokens. Return how many sequences fit (rounded down).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_sequences, [
    ((80, 16, 0.125, 8192), 54),
    ((80, 16, 0.125, 2048), 218),
    ((24, 16, 0.125, 8192), 5),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Subtract and divide step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Floor division in one expression, clamped at 0.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Subtract and divide step by step.

```python
def max_sequences(gpu_gb, weights_gb, kv_mb_per_token, tokens):
    free_gb = gpu_gb * 0.9 - weights_gb
    per_seq_gb = kv_mb_per_token * tokens / 1000
    n = int(free_gb // per_seq_gb)
    if n < 0:
        n = 0
    return n
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Floor division in one expression, clamped at 0.

```python
def max_sequences(gpu_gb, weights_gb, kv_mb_per_token, tokens):
    return max(0, int((gpu_gb * 0.9 - weights_gb) // (kv_mb_per_token * tokens / 1000)))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Subtract and divide step by step. That costs O(1) time and O(1) space."

"I can do better: Floor division in one expression, clamped at 0. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/how-many-users-at-once)

<a id="q10"></a>
### Q10. Speculative Decoding: Expected Speed-Up

🟡 Interview standard · *Draft and verify*

A small draft model proposes `k` tokens; the big model checks them in one step. Each is accepted with probability `a` until the first rejection, and the big model always adds one token of its own. Expected tokens per big-model step = (1 − a^(k+1)) / (1 − a). Return it for each k in `ks`, rounded to 2.

**Examples**

- `a = 0.8, ks = [1, 2, 4, 8]` → `[1.8, 2.44, 3.36, 4.33]`
- `a = 0.5, ks = [1, 4, 8]` → `[1.5, 1.94, 2.0]`

In [ ]:
def spec_tokens(a, ks):
    """
    A small draft model proposes k tokens; the big model checks them in one
    step. Each is accepted with probability a until the first rejection, and the
    big model always adds one token of its own. Expected tokens per big-model
    step = (1 − a^(k+1)) / (1 − a). Return it for each k in ks, rounded to 2.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(spec_tokens, [
    ((0.8, [1, 2, 4, 8]), [1.8, 2.44, 3.36, 4.33]),
    ((0.5, [1, 4, 8]), [1.5, 1.94, 2.0]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sum the probabilities that the first j tokens are all accepted, j = 0..k.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The geometric-series closed form.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Sum the probabilities that the first j tokens are all accepted, j = 0..k.

```python
def spec_tokens(a, ks):
    out = []
    for k in ks:
        total = 0
        for j in range(k + 1):
            total += a ** j
        out.append(round(total, 2))
    return out
```

**Cost:** O(k) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** The geometric-series closed form.

```python
def spec_tokens(a, ks):
    return [round((1 - a ** (k + 1)) / (1 - a), 2) for k in ks]
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sum the probabilities that the first j tokens are all accepted, j = 0..k. That costs O(k) time and O(1) space."

"I can do better: The geometric-series closed form. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/speculative-decoding-expected-speed-up)

<a id="q11"></a>
### Q11. Verify a Draft

🟡 Interview standard · *Accepting the matching prefix*

Greedy speculative decoding: keep the draft's tokens while they match what the big model would produce, then take the big model's token at the first mismatch (or one extra token if all match). Return `(accepted draft tokens, tokens emitted this step)`.

**Examples**

- `draft = ['The', 'cat', 'sat', 'on'], target = ['The', 'cat', 'is', 'on', 'the']` → `(2, ['The', 'cat', 'is'])`
- `draft = ['a', 'b'], target = ['a', 'b', 'c']` → `(2, ['a', 'b', 'c'])`
- `draft = ['x'], target = ['y', 'z']` → `(0, ['y'])`

In [ ]:
def verify_draft(draft, target):
    """
    Greedy speculative decoding: keep the draft's tokens while they match what
    the big model would produce, then take the big model's token at the first
    mismatch (or one extra token if all match). Return (accepted draft tokens,
    tokens emitted this step).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(verify_draft, [
    ((['The', 'cat', 'sat', 'on'], ['The', 'cat', 'is', 'on', 'the']), (2, ['The', 'cat', 'is'])),
    ((['a', 'b'], ['a', 'b', 'c']), (2, ['a', 'b', 'c'])),
    ((['x'], ['y', 'z']), (0, ['y'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk both lists with an index.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`itertools.takewhile` over the zipped pair.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk both lists with an index.

```python
def verify_draft(draft, target):
    n = 0
    while n < len(draft) and draft[n] == target[n]:
        n += 1
    return (n, target[:n + 1])
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `itertools.takewhile` over the zipped pair.

```python
def verify_draft(draft, target):
    from itertools import takewhile
    n = sum(1 for _ in takewhile(lambda p: p[0] == p[1], zip(draft, target)))
    return (n, target[:n + 1])
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk both lists with an index. That costs O(k) time and O(k) space."

"I can do better: `itertools.takewhile` over the zipped pair. That's O(k) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/verify-a-draft)

<a id="q12"></a>
### Q12. Quantise to int8

🟡 Interview standard · *Smaller and faster*

Symmetric int8 quantisation: scale = max |w| / 127, q = round(w / scale), restored = q × scale. Return `(q values, largest absolute error rounded to 4)`.

**Examples**

- `weights = [0.42, -1.27, 0.03, 0.88, -0.51, 2.31, -0.07, 0.66]` → `([23, -70, 2, 48, -28, 127, -4, 36], 0.0069)`

In [ ]:
def int8(weights):
    """
    Symmetric int8 quantisation: scale = max |w| / 127, q = round(w / scale),
    restored = q × scale. Return (q values, largest absolute error rounded to
    4).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(int8, [
    (([0.42, -1.27, 0.03, 0.88, -0.51, 2.31, -0.07, 0.66],), ([23, -70, 2, 48, -28, 127, -4, 36], 0.0069)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over weights.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

NumPy: `np.round`, `np.abs`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop over weights.

```python
def int8(weights):
    scale = max(abs(w) for w in weights) / 127
    q, err = [], 0
    for w in weights:
        v = round(w / scale)
        q.append(v)
        err = max(err, abs(w - v * scale))
    return (q, round(err, 4))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** NumPy: `np.round`, `np.abs`.

```python
def int8(weights):
    w = np.array(weights)
    scale = np.abs(w).max() / 127
    q = np.round(w / scale)
    return (q.astype(int).tolist(), round(float(np.abs(w - q * scale).max()), 4))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over weights. That costs O(n) time and O(n) space."

"I can do better: NumPy: `np.round`, `np.abs`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/quantise-to-int8)

<a id="q13"></a>
### Q13. Why Group-Wise Scales Help

🟡 Interview standard · *Outliers in quantisation*

Quantise to int4 (levels −7…7, scale = max|w|/7) either with **one** scale for all weights or a separate scale per `group` consecutive weights. Return `(mean absolute error per-tensor, per-group)`, rounded to 4. One big weight ruins a shared scale.

**Examples**

- `weights = [0.42, -1.27, 0.03, 0.88, -0.51, 2.31, -0.07, 0.66], group = 4` → `(0.0625, 0.0418)`
- `weights = [0.42, -1.27, 0.03, 0.88, -0.51, 2.31, -0.07, 0.66], group = 2` → `(0.0625, 0.0327)`

In [ ]:
def group_quant(weights, group):
    """
    Quantise to int4 (levels −7…7, scale = max|w|/7) either with one scale for
    all weights or a separate scale per group consecutive weights. Return (mean
    absolute error per-tensor, per-group), rounded to 4. One big weight ruins a
    shared scale.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(group_quant, [
    (([0.42, -1.27, 0.03, 0.88, -0.51, 2.31, -0.07, 0.66], 4), (0.0625, 0.0418)),
    (([0.42, -1.27, 0.03, 0.88, -0.51, 2.31, -0.07, 0.66], 2), (0.0625, 0.0327)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A helper that quantises a list, used once and per group.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

NumPy reshape into groups and a per-row max.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A helper that quantises a list, used once and per group.

```python
def group_quant(weights, group):
    def err(ws):
        scale = max(abs(w) for w in ws) / 7
        return [abs(w - round(w / scale) * scale) for w in ws]
    whole = err(weights)
    grouped = []
    for i in range(0, len(weights), group):
        grouped += err(weights[i:i + group])
    return (round(sum(whole) / len(whole), 4), round(sum(grouped) / len(grouped), 4))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** NumPy reshape into groups and a per-row max.

```python
def group_quant(weights, group):
    w = np.array(weights)

    def mae(m):
        scale = np.abs(m).max(axis=-1, keepdims=True) / 7
        return np.abs(m - np.round(m / scale) * scale).mean()
    return (round(float(mae(w[None, :])), 4), round(float(mae(w.reshape(-1, group))), 4))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A helper that quantises a list, used once and per group. That costs O(n) time and O(n) space."

"I can do better: NumPy reshape into groups and a per-row max. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `weights = [0.42, -1.27, 0.03, 0.88, -0.51, 2.31, -0.07, 0.66], group = 4`

1. mae() returns 0.0625
2. mae() returns 0.0417857142857143
3. group\_quant() returns (0.0625, 0.0418)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/why-group-wise-scales-help)

<a id="q14"></a>
### Q14. Paged KV Memory

🟡 Interview standard · *PagedAttention*

Old servers reserved `max_len` tokens of KV memory per sequence up front. Paged allocation uses blocks of `block` tokens, only as needed. Return `(tokens reserved up front, tokens reserved with paging, % wasted up front)` for sequences of the given lengths.

**Examples**

- `lengths = [300, 1200, 50, 4096, 700], max_len = 4096, block = 16` → `(20480, 6368, 69.0)`

In [ ]:
def paged(lengths, max_len, block):
    """
    Old servers reserved max_len tokens of KV memory per sequence up front.
    Paged allocation uses blocks of block tokens, only as needed. Return (tokens
    reserved up front, tokens reserved with paging, % wasted up front) for
    sequences of the given lengths.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(paged, [
    (([300, 1200, 50, 4096, 700], 4096, 16), (20480, 6368, 69.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ceiling division per sequence in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sums with `-(-L // block) * block`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Ceiling division per sequence in a loop.

```python
def paged(lengths, max_len, block):
    upfront = max_len * len(lengths)
    paged_total = 0
    for L in lengths:
        blocks = L // block
        if L % block:
            blocks += 1
        paged_total += blocks * block
    used = sum(lengths)
    return (upfront, paged_total, round((upfront - used) / upfront * 100, 1))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Sums with `-(-L // block) * block`.

```python
def paged(lengths, max_len, block):
    upfront = max_len * len(lengths)
    return (upfront, sum(-(-L // block) * block for L in lengths), round((upfront - sum(lengths)) / upfront * 100, 1))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ceiling division per sequence in a loop. That costs O(n) time and O(1) space."

"I can do better: Sums with `-(-L // block) * block`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/paged-kv-memory)

<a id="q15"></a>
### Q15. How Many GPUs for the Traffic?

🟡 Interview standard · *Little's law*

Requests arrive at `rps` per second and each takes `secs` seconds on a server that handles `per_gpu` requests at once. Keep utilisation at most `util`. Return the number of GPUs (round up).

**Examples**

- `rps = 20, secs = 6, per_gpu = 32, util = 0.7` → `6`
- `rps = 5, secs = 2, per_gpu = 64, util = 0.7` → `1`

In [ ]:
def gpus_needed(rps, secs, per_gpu, util):
    """
    Requests arrive at rps per second and each takes secs seconds on a server
    that handles per_gpu requests at once. Keep utilisation at most util. Return
    the number of GPUs (round up).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(gpus_needed, [
    ((20, 6, 32, 0.7), 6),
    ((5, 2, 64, 0.7), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

In-flight = rps × secs, then divide by usable capacity per GPU.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`math.ceil` of one expression.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** In-flight = rps × secs, then divide by usable capacity per GPU.

```python
def gpus_needed(rps, secs, per_gpu, util):
    in_flight = rps * secs
    usable = per_gpu * util
    n = 1
    while n * usable < in_flight:
        n += 1
    return n
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `math.ceil` of one expression.

```python
def gpus_needed(rps, secs, per_gpu, util):
    return max(1, math.ceil(round(rps * secs / (per_gpu * util), 9)))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: In-flight = rps × secs, then divide by usable capacity per GPU. That costs O(1) time and O(1) space."

"I can do better: `math.ceil` of one expression. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/how-many-gpus-for-the-traffic)

<a id="q16"></a>
### Q16. Self-Host or Use an API?

🟡 Interview standard · *Break-even volume*

Self-hosting costs a fixed `fixed_month` per month (GPUs, engineers) and the API costs `api_per_m` dollars per million tokens. Return the monthly volume (millions of tokens, rounded up) above which self-hosting is cheaper, and the cheaper option at `volume` million tokens.

**Examples**

- `fixed_month = 12000, api_per_m = 6.0, volume = 500` → `(2000, 'api')`
- `fixed_month = 12000, api_per_m = 6.0, volume = 5000` → `(2000, 'self-host')`

In [ ]:
def break_even(fixed_month, api_per_m, volume):
    """
    Self-hosting costs a fixed fixed_month per month (GPUs, engineers) and the
    API costs api_per_m dollars per million tokens. Return the monthly volume
    (millions of tokens, rounded up) above which self-hosting is cheaper, and
    the cheaper option at volume million tokens.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(break_even, [
    ((12000, 6.0, 500), (2000, 'api')),
    ((12000, 6.0, 5000), (2000, 'self-host')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Divide, then compare costs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One expression each.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Divide, then compare costs.

```python
def break_even(fixed_month, api_per_m, volume):
    even = fixed_month / api_per_m
    if even != int(even):
        even = int(even) + 1
    choice = "self-host" if volume * api_per_m > fixed_month else "api"
    return (int(even), choice)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One expression each.

```python
def break_even(fixed_month, api_per_m, volume):
    return (math.ceil(fixed_month / api_per_m), "self-host" if volume * api_per_m > fixed_month else "api")
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Divide, then compare costs. That costs O(1) time and O(1) space."

"I can do better: One expression each. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/self-host-or-use-an-api)

<a id="q17"></a>
### Q17. Shortest Job First

🟡 Interview standard · *Scheduling*

All requests arrive together and run one at a time; `lengths` are their run times. Return the average waiting time (rounded to 2) for first-come-first-served and for shortest-job-first.

**Examples**

- `lengths = [40, 2, 15, 3, 8]` → `(39.8, 9.6)`

In [ ]:
def avg_wait(lengths):
    """
    All requests arrive together and run one at a time; lengths are their run
    times. Return the average waiting time (rounded to 2) for first-come-first-
    served and for shortest-job-first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(avg_wait, [
    (([40, 2, 15, 3, 8],), (39.8, 9.6)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Accumulate start times in a loop for each order.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`itertools.accumulate` gives each job's start time.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Accumulate start times in a loop for each order.

```python
def avg_wait(lengths):
    def wait(order):
        t, total = 0, 0
        for L in order:
            total += t
            t += L
        return total / len(order)
    return (round(wait(lengths), 2), round(wait(sorted(lengths)), 2))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `itertools.accumulate` gives each job's start time.

```python
def avg_wait(lengths):
    from itertools import accumulate
    wait = lambda order: sum(accumulate([0] + order[:-1])) / len(order)
    return (round(wait(lengths), 2), round(wait(sorted(lengths)), 2))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Accumulate start times in a loop for each order. That costs O(n log n) time and O(n) space."

"I can do better: `itertools.accumulate` gives each job's start time. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/shortest-job-first)

<a id="q18"></a>
### Q18. A Token Bucket Rate Limiter

🟡 Interview standard · *Tokens per minute*

Allow at most `capacity` tokens in the bucket, refilled at `rate` tokens per second (starting full). Each request `(time, tokens)` is accepted if the bucket holds enough, and then those tokens are removed. Return accept/reject booleans.

**Examples**

- `requests = [(0, 600), (1, 600), (2, 300), (10, 800), (10.5, 300)], capacity = 1000, rate = 50` → `[True, False, True, False, True]`

In [ ]:
def token_bucket(requests, capacity, rate):
    """
    Allow at most capacity tokens in the bucket, refilled at rate tokens per
    second (starting full). Each request (time, tokens) is accepted if the
    bucket holds enough, and then those tokens are removed. Return accept/reject
    booleans.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(token_bucket, [
    (([(0, 600), (1, 600), (2, 300), (10, 800), (10.5, 300)], 1000, 50), [True, False, True, False, True]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track the level and the last time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, with `min` for the refill.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Track the level and the last time.

```python
def token_bucket(requests, capacity, rate):
    level, last, out = capacity, 0, []
    for t, need in requests:
        level = level + (t - last) * rate
        if level > capacity:
            level = capacity
        last = t
        if need <= level:
            level -= need
            out.append(True)
        else:
            out.append(False)
    return out
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, with `min` for the refill.

```python
def token_bucket(requests, capacity, rate):
    level, last, out = capacity, 0, []
    for t, need in requests:
        level, last = min(capacity, level + (t - last) * rate), t
        ok = need <= level
        level -= need if ok else 0
        out.append(ok)
    return out
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track the level and the last time. That costs O(n) time and O(1) space."

"I can do better: Same, with `min` for the refill. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/a-token-bucket-rate-limiter)

<a id="q19"></a>
### Q19. Longest Context That Fits

🟡 Interview standard · *Memory budgeting*

With `free_gb` GB left after weights, and KV cost `kv_kb_per_token` KB per token, return the longest context (tokens, rounded down to a multiple of 1,024) each of `batch` sequences can have.

**Examples**

- `free_gb = 40, kv_kb_per_token = 128, batch = 8` → `38912`
- `free_gb = 40, kv_kb_per_token = 128, batch = 1` → `312320`

In [ ]:
def max_context(free_gb, kv_kb_per_token, batch):
    """
    With free_gb GB left after weights, and KV cost kv_kb_per_token KB per
    token, return the longest context (tokens, rounded down to a multiple of
    1,024) each of batch sequences can have.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_context, [
    ((40, 128, 8), 38912),
    ((40, 128, 1), 312320),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute tokens per sequence, then round down.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Floor division twice.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compute tokens per sequence, then round down.

```python
def max_context(free_gb, kv_kb_per_token, batch):
    tokens = free_gb * 1e6 / kv_kb_per_token / batch
    tokens = int(tokens)
    return tokens - tokens % 1024
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Floor division twice.

```python
def max_context(free_gb, kv_kb_per_token, batch):
    return int(free_gb * 1e6 // (kv_kb_per_token * batch)) // 1024 * 1024
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute tokens per sequence, then round down. That costs O(1) time and O(1) space."

"I can do better: Floor division twice. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/longest-context-that-fits)

<a id="q20"></a>
### Q20. Static vs Continuous Batching

🔴 Stretch · *Why vLLM is fast*

Requests need `lengths[i]` decode steps; the GPU runs `slots` sequences per step. **Static**: take requests in groups of `slots`; a group takes as long as its longest request. **Continuous**: whenever a sequence finishes, the next waiting request takes its slot. Return `(static steps, continuous steps)`.

**Examples**

- `lengths = [10, 200, 15, 30, 190, 20, 25, 12], slots = 2` → `(445, 257)`
- `lengths = [10, 200, 15, 30, 190, 20, 25, 12], slots = 4` → `(390, 200)`

In [ ]:
def batching_steps(lengths, slots):
    """
    Requests need lengths[i] decode steps; the GPU runs slots sequences per
    step. Static: take requests in groups of slots; a group takes as long as its
    longest request. Continuous: whenever a sequence finishes, the next waiting
    request takes its slot. Return (static steps, continuous steps).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(batching_steps, [
    (([10, 200, 15, 30, 190, 20, 25, 12], 2), (445, 257)),
    (([10, 200, 15, 30, 190, 20, 25, 12], 4), (390, 200)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Static: sum of each group's max. Continuous: simulate slots freeing up.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Continuous is a min-heap of slot finish times (like a worker pool).

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Static: sum of each group's max. Continuous: simulate slots freeing up.

```python
def batching_steps(lengths, slots):
    static = 0
    for i in range(0, len(lengths), slots):
        static += max(lengths[i:i + slots])
    finish = [0] * slots
    for L in lengths:
        j = finish.index(min(finish))
        finish[j] += L
    return (static, max(finish))
```

**Cost:** O(n · slots) time, O(slots) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Continuous is a min-heap of slot finish times (like a worker pool).

```python
def batching_steps(lengths, slots):
    import heapq
    static = sum(max(lengths[i:i + slots]) for i in range(0, len(lengths), slots))
    heap = [0] * slots
    for L in lengths:
        heapq.heappush(heap, heapq.heappop(heap) + L)
    return (static, max(heap))
```

**Cost:** O(n log slots) time, O(slots) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Static: sum of each group's max. Continuous: simulate slots freeing up. That costs O(n · slots) time and O(slots) space."

"I can do better: Continuous is a min-heap of slot finish times (like a worker pool). That's O(n log slots) time and O(slots) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/serving/static-vs-continuous-batching)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What is the KV cache?</b></summary>

Stored key and value vectors for all previous tokens in every layer, so each new token only computes its own. It trades memory for speed and grows with context length and batch size.

</details>

<details>
<summary><b>2. Continuous vs static batching?</b></summary>

Static batching runs a fixed group until its longest sequence finishes. Continuous batching adds new requests as soon as any sequence finishes, keeping the GPU full and raising throughput.

</details>

<details>
<summary><b>3. How does speculative decoding keep the same output?</b></summary>

The big model verifies the draft's tokens and only accepts those it would have produced itself (or, with sampling, accepts them with a probability that preserves its distribution), so quality is unchanged while several tokens can be emitted per big-model step.

</details>

<details>
<summary><b>4. What does quantisation cost you?</b></summary>

Some accuracy, especially at 4 bits or below and on reasoning-heavy tasks. Measure on your own evaluation set; per-group methods like GPTQ and AWQ keep the loss small.

</details>

<details>
<summary><b>5. When does self-hosting beat an API?</b></summary>

At high, steady volume where fixed GPU and engineering costs spread over many tokens, or when data residency, latency control, offline use or a custom fine-tuned model require it.

</details>

<details>
<summary><b>6. What is PagedAttention?</b></summary>

Managing the KV cache in fixed-size blocks allocated on demand (like virtual memory pages), avoiding the waste of reserving maximum length per sequence and enabling prefix sharing.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: API or your own GPUs?
- [ ] I can explain: Memory: weights and the KV cache
- [ ] I can explain: Batching, prefill and decode
- [ ] I can explain: Making each token cheaper
- [ ] I can explain: Running it in production

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Self-Host or Use an API?
- [ ] [Q17](#q17) Shortest Job First
- [ ] [Q18](#q18) A Token Bucket Rate Limiter
- [ ] [Q19](#q19) Longest Context That Fits
- [ ] [Q20](#q20) Static vs Continuous Batching

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.